Ce notebook a pour objectif de construire des modèles de Machine Learning capables de prédire le prix des maisons à partir de leurs caractéristiques principales.

Deux modèles sont comparés :

une régression linéaire (modèle simple et interprétable)

un Random Forest (modèle plus complexe et performant)

 1. Chargement et préparation des données

Le dataset est rechargé via le pipeline Kaggle afin de garantir la reproductibilité du projet.

Ensuite, seules certaines variables sont sélectionnées :

surface habitable (GrLivArea)

qualité générale (OverallQual)

nombre de places de garage (GarageCars)

surface du sous-sol (TotalBsmtSF)

année de construction (YearBuilt)

La variable cible est :

SalePrice (prix de la maison)

Les lignes contenant des valeurs manquantes sont supprimées afin de garantir la qualité du modèle.

 2. Séparation entraînement / test

Les données sont divisées en deux parties :

80% entraînement

20% test

Cela permet d’évaluer les performances du modèle sur des données qu’il n’a jamais vues.

 3. Modèle : Régression Linéaire

Un premier modèle simple est entraîné : la régression linéaire.

 Résultats :

MAE : erreur moyenne absolue

RMSE : erreur quadratique moyenne

R² : qualité d’explication du modèle

Ce modèle permet une première approximation des prix, mais reste limité pour capturer des relations complexes.

 4. Modèle : Random Forest

Un modèle plus avancé est ensuite utilisé : Random Forest Regressor.

Ce modèle est basé sur plusieurs arbres de décision combinés, ce qui permet :

de mieux capturer les relations non linéaires

d’améliorer la précision des prédictions

de réduire le surapprentissage

 Résultats :

Les métriques montrent généralement de meilleures performances que la régression linéaire, notamment en termes de précision et de robustesse.

 5. Test de prédiction

Un exemple de maison est utilisé pour tester le modèle :

surface : 2000

qualité : 8

garage : 2 places

sous-sol : 1000

année de construction : 2005

Le modèle Random Forest estime le prix de cette maison en fonction des patterns appris dans les données.

In [1]:
import pandas as pd
import os
import zipfile

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np

# ==============================
# LOAD DATASET (SAME PIPELINE)
# ==============================

os.system(
    "kaggle competitions download -c house-prices-advanced-regression-techniques"
)

with zipfile.ZipFile(
    "house-prices-advanced-regression-techniques.zip",
    "r"
) as zip_ref:
    zip_ref.extractall("data/raw")

df = pd.read_csv("data/raw/train.csv")

# ==============================
# FEATURE SELECTION
# ==============================

features = [
    "GrLivArea",
    "OverallQual",
    "GarageCars",
    "TotalBsmtSF",
    "YearBuilt"
]

target = "SalePrice"

# ==============================
# CLEAN DATA
# ==============================

df_model = df[features + [target]].dropna()

X = df_model[features]
y = df_model[target]

# ==============================
# TRAIN TEST SPLIT
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Train size :", X_train.shape)
print("Test size :", X_test.shape)

# ==============================
# LINEAR REGRESSION
# ==============================

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_preds = lr_model.predict(X_test)

mae_lr = mean_absolute_error(y_test, lr_preds)
rmse_lr = np.sqrt(mean_squared_error(y_test, lr_preds))
r2_lr = r2_score(y_test, lr_preds)

print("\n📊 Linear Regression")
print("MAE :", mae_lr)
print("RMSE :", rmse_lr)
print("R2 :", r2_lr)

# ==============================
# RANDOM FOREST
# ==============================

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)

mae_rf = mean_absolute_error(y_test, rf_preds)
rmse_rf = np.sqrt(mean_squared_error(y_test, rf_preds))
r2_rf = r2_score(y_test, rf_preds)

print("\n🌲 Random Forest")
print("MAE :", mae_rf)
print("RMSE :", rmse_rf)
print("R2 :", r2_rf)

# ==============================
# TEST PREDICTION
# ==============================

sample_house = pd.DataFrame([[
    2000,
    8,
    2,
    1000,
    2005
]], columns=features)

prediction = rf_model.predict(sample_house)

print("\n🏠 Prix estimé :", prediction[0])

Train size : (1168, 5)
Test size : (292, 5)

📊 Linear Regression
MAE : 25414.72540285201
RMSE : 39763.295265780245
R2 : 0.7938653966356598

🌲 Random Forest
MAE : 19186.15421194825
RMSE : 29367.781572076077
R2 : 0.8875580293238329

🏠 Prix estimé : 253382.275
